# Okay lets try this again

In [1]:
# to generate all signed permutations in S_n^C and one (not necessarily reduced) word for those permutations

# I know there's a better way to do this built into Sage but because of the version of Sage I'm running & my lack of
# understanding of Sage, I couldn't get it to work

# Note: that word is found by taking a reduced word for the "flat" permutation (the permutation in S_n that is the same
# as the permutaion in S_n^C but without negatives) and then appending lists of the form [i,i-1,...,1,0,1,...,i-1,i] to
# negate the necessary entries

# Getting all the signed perms in S_n^C

# given n, find all reduced words FOR THE TYPE A PERMUTATIONS IN S_n
def get_rws(n):
    permDict = {}
    for perm in Permutations(n):
        permDict[perm] = perm.reduced_word()
    return permDict
# test
print(f"reduced words in S_3: {get_rws(3)}")
print("\n")

# given n, find all vectors in {0,1}^n
from itertools import product

def binary_vec(n):
    return product([0, 1], repeat=n)
# test
print(f"vectors for n=3: {binary_vec(3)}")
for vec in binary_vec(3):
    print(vec)
print("\n")

# given a word L and a vector v, return the word L but for every 1 in v, that index in L is now negative
def negate_by_vector(L, v):
    if len(L) != len(v):
        raise ValueError("L and v must have the same length.")

    return [-x if b == 1 else x for x, b in zip(L, v)]

# get all the signed permutations in S_n^C as dictionaries with:
    # the "flat permutation" saved as 'flat'
    # the vector v giving the signs saved as 'vector'
    # the full signed permutation saved as 'signed perm'
    # the (not necessarily reduced) word saved as 'word'
def get_signed_rws(n):
    signed_perms = []
    permDict = get_rws(n)
    vectors = list(binary_vec(n))
    for perm in permDict:
        for vector in vectors:
            signed_perms.append({'flat':perm,
                                 'vector':vector,
                                 'signed perm':negate_by_vector(perm,vector),
                                 'word':extend_list(vector,permDict[perm])})
    return signed_perms

# function to find flat(w) and vector(w) given w in S_n^C
def abs_and_sign_vector(L):
    """
    Input:
        L -- a list of integers.

    Output:
        (M, v), where
          M[i] = abs(L[i]),
          v[i] = 0 if L[i] >= 0,
                 1 if L[i] < 0.
    """
    M = [abs(x) for x in L]
    v = vector([1 if x < 0 else 0 for x in L])
    return (M, v)

# finding the (not necessarily reduced) words

# given some integer i, return the list [i,i-1,...,1,0,1,...i-1,i]
def symmetric_list(i):
    if i < 0:
        raise ValueError("i must be a nonnegative integer")
    return list(range(i, -1, -1)) + list(range(1, i + 1))
# test
print(f"list to add on if the entry in index 3 is negative: {symmetric_list(3)}")
print("\n")

# given a binary vector v and a list L, for every 1 in v, if it's at index j append the list [j,j-1,...,1,0,1,...,j-1,j]
def extend_list(v, L):
    out = list(L)   # make a copy

    for j, x in enumerate(v):
        if x == 1:
            out.extend(symmetric_list(j))

    return out
    
# test
L = [2,1,2]
v = (0,1,1)
print(f"a (nonreduced) word for [3,-2,-1] (reduced_word(flat(w))=[2,1,2], sign(w)=(0,1,1)): {extend_list(v,L)}")
print(f"flat and sign of [3,-2,-1]: {abs_and_sign_vector([3,-2,-1])}")
print("\n")

#test 2
L = [2,1]
v = (0,1,1,0)
print(f"a (nonreduced) word for [3,-1,-2,4] (reduced_word(flat(w))=[2,1], sign(w)=(0,1,1,0)): {extend_list(v,L)}")

reduced words in S_3: {[1, 2, 3]: [], [1, 3, 2]: [2], [2, 1, 3]: [1], [2, 3, 1]: [1, 2], [3, 1, 2]: [2, 1], [3, 2, 1]: [2, 1, 2]}


vectors for n=3: <itertools.product object at 0x7f1e488f2580>
(0, 0, 0)
(0, 0, 1)
(0, 1, 0)
(0, 1, 1)
(1, 0, 0)
(1, 0, 1)
(1, 1, 0)
(1, 1, 1)


list to add on if the entry in index 3 is negative: [3, 2, 1, 0, 1, 2, 3]


a (nonreduced) word for [3,-2,-1] (reduced_word(flat(w))=[2,1,2], sign(w)=(0,1,1)): [2, 1, 2, 1, 0, 1, 2, 1, 0, 1, 2]
flat and sign of [3,-2,-1]: ([3, 2, 1], (0, 1, 1))


a (nonreduced) word for [3,-1,-2,4] (reduced_word(flat(w))=[2,1], sign(w)=(0,1,1,0)): [2, 1, 1, 0, 1, 2, 1, 0, 1, 2]


In [2]:
# Finding "good" reduced words
# prompt given to GPT-5.5:
"""
Write me a SageMath function that can do the following: The function should take in a list of non-negative integers.
Then, it should find all "equivalent" lists of the same length or shorter, where two lists are equivalent if any of
these conditions are met:
    (A) If List 1 has the consecutive sublist [i,i] for some integer i, and List 2 is exactly the same as List 1
        but with this sublist removed;
    (B) If List 1 has the consecutive sublist [i,j] for |i-j|>1, and List 2 is the same as List 1 but with [j,i]
        instead of List 1's [i,j];
    (C) If List 1 has the consecutive sublist [i,i+1,i] for some integer i>0, and List 2 is the same as List 1
        but with [i+1,i,i+1] instead of List 1's [i,i+1,i];
    (D) If List 1 has the consecutive sublist [0,1,0,1] and List 2 is the same as List 1 but with [1,0,1,0] instead
        of List 1's [0,1,0,1].
Then, after the function has found all of the equivalent lists, I want it to only look at those equivalent lists of
the shortest length. From these, the function should pick out one of these lists that has the longest possible "strictly
increasing suffix", meaning that it ends with the longest possible consecutive sublist where the elements of the sublist
are in strictly increasing order. (For example, between the two lists [1,2,1] and [2,1,2], [2,1,2] has the longer
strictly increasing suffix ([1,2] as opposed to [1,2,1]'s strictly increasing suffix of [1])). The function should then
output the chosen list. (This is based off the Type C braid relations, but I specifically want the function be to
constructed in this way, even though there may be more efficient ways to achieve the same or similar results using Sage's
Weyl or Coxeter group capabilities.)
"""

# then debugged from there

from collections import deque

def increasing_suffix_length(word):
    """
    Length of the longest strictly increasing suffix.
    """
    if len(word) == 0:
        return 0

    k = 1
    for i in range(len(word)-2, -1, -1):
        if word[i] < word[i+1]:
            k += 1
        else:
            break
    return k


def neighbors(word):
    w = list(word)
    n = len(w)
    out = []

    # (A)
    for i in range(n-1):
        if w[i] == w[i+1]:
            new = tuple(w[:i] + w[i+2:])
            # print("A", i, ":", w, "->", list(new))
            out.append(new)

    # (B)
    for i in range(n-1):
        if abs(w[i]-w[i+1]) > 1:
            new = w[:]
            new[i], new[i+1] = new[i+1], new[i]
            # print("B", i, ":", w, "->", new)
            out.append(tuple(new))

    # (C): braid [i,i+1,i] <-> [i+1,i,i+1], i>0
    for i in range(n-2):
        a, b, c = w[i:i+3]
    
        # [i,i+1,i] -> [i+1,i,i+1]
        if a > 0 and b == a+1 and c == a:
            new = w[:]
            new[i:i+3] = [a+1, a, a+1]
            out.append(tuple(new))
    
        # [i+1,i,i+1] -> [i,i+1,i]
        elif a > 1 and b == a-1 and c == a:
            new = w[:]
            new[i:i+3] = [a-1, a, a-1]
            out.append(tuple(new))

    # (D)
    for i in range(n-3):
        block = w[i:i+4]

        if block == [0,1,0,1]:
            new = w[:]
            new[i:i+4] = [1,0,1,0]
            # print("D forward", i, ":", w, "->", new)
            out.append(tuple(new))

        elif block == [1,0,1,0]:
            new = w[:]
            new[i:i+4] = [0,1,0,1]
            # print("D inverse", i, ":", w, "->", new)
            out.append(tuple(new))

    return out

# test
w = [0,1,0,2,1,2,2]
print(f"neighbors of {w}: {neighbors(w)}")
print("\n")

from collections import deque

def canonical_typeC_word(word, debug=False):
    """
    Return the representative requested in the question.

    Among all equivalent words:
      1. keep only those of minimal length;
      2. choose one with the longest strictly increasing suffix.

    If there is still a tie, choose the lexicographically smallest
    one to make the output deterministic.

    If debug=True, also print the sequence of words leading from the
    input to the chosen representative.
    """

    start = tuple(word)

    visited = {start}
    parent = {start: None}
    Q = deque([start])

    while Q:
        v = Q.popleft()
        for u in neighbors(v):
            if u not in visited:
                visited.add(u)
                parent[u] = v
                Q.append(u)

    # shortest length
    minlen = min(len(w) for w in visited)
    shortest = [w for w in visited if len(w) == minlen]

    # maximize increasing suffix length
    best_suffix = max(increasing_suffix_length(w) for w in shortest)
    candidates = [w for w in shortest
                  if increasing_suffix_length(w) == best_suffix]

    # deterministic tie-break
    best = min(candidates)

    if debug:
        path = []
        x = best
        while x is not None:
            path.append(x)
            x = parent[x]
        path.reverse()

        print("Transformation path:")
        for i, w in enumerate(path):
            print(f"{i:2d}: {list(w)}")

    return list(best)

# test
t_pre = [2, 1, 1, 0, 1, 2, 1, 0, 1, 2]
print(f"reduced word for [3,-1,-2,4] w/ max'l strictly inc'g suffix: {canonical_typeC_word(t_pre)}")

# test 2
t_pre = [0,1,0,1,2,1,0,1,2]
print(f"reduced word for [-1,-2,-3] w/ max'l strictly inc'g suffix: {canonical_typeC_word(t_pre)}")

# test 3
t_pre = [2, 1, 2, 1, 0, 1, 2, 1, 0, 1, 2]
print(f"reduced word for [3,-2,-1] w/ max'l strictly inc'g suffix: {canonical_typeC_word(t_pre)}")

neighbors of [0, 1, 0, 2, 1, 2, 2]: [(0, 1, 0, 2, 1), (0, 1, 2, 0, 1, 2, 2), (0, 1, 0, 1, 2, 1, 2)]


reduced word for [3,-1,-2,4] w/ max'l strictly inc'g suffix: [0, 1, 2, 0, 1, 2]
reduced word for [-1,-2,-3] w/ max'l strictly inc'g suffix: [0, 1, 0, 1, 2, 1, 0, 1, 2]
reduced word for [3,-2,-1] w/ max'l strictly inc'g suffix: [0, 1, 2, 0, 1]


In [3]:
# # Finding "good" reduced words
# # prompt given to GPT-5.5:
# """
# Write me a SageMath function that can do the following: The function should take in a list of non-negative integers.
# Then, it should find all "equivalent" lists of the same length or shorter, where two lists are equivalent if any of
# these conditions are met:
#     (A) If List 1 has the consecutive sublist [i,i] for some integer i, and List 2 is exactly the same as List 1
#         but with this sublist removed;
#     (B) If List 1 has the consecutive sublist [i,j] for |i-j|>1, and List 2 is the same as List 1 but with [j,i]
#         instead of List 1's [i,j];
#     (C) If List 1 has the consecutive sublist [i,i+1,i] for some integer i>0, and List 2 is the same as List 1
#         but with [i+1,i,i+1] instead of List 1's [i,i+1,i];
#     (D) If List 1 has the consecutive sublist [0,1,0,1] and List 2 is the same as List 1 but with [1,0,1,0] instead
#         of List 1's [0,1,0,1].
# Then, after the function has found all of the equivalent lists, I want it to only look at those equivalent lists of
# the shortest length. From these, the function should pick out one of these lists that has the longest possible "strictly
# increasing suffix", meaning that it ends with the longest possible consecutive sublist where the elements of the sublist
# are in strictly increasing order. (For example, between the two lists [1,2,1] and [2,1,2], [2,1,2] has the longer
# strictly increasing suffix ([1,2] as opposed to [1,2,1]'s strictly increasing suffix of [1])). The function should then
# output the chosen list. (This is based off the Type C braid relations, but I specifically want the function be to
# constructed in this way, even though there may be more efficient ways to achieve the same or similar results using Sage's
# Weyl or Coxeter group capabilities.)
# """

# from collections import deque

# def increasing_suffix_length(word):
#     """
#     Length of the longest strictly increasing suffix.
#     """
#     if len(word) == 0:
#         return 0

#     k = 1
#     for i in range(len(word)-2, -1, -1):
#         if word[i] < word[i+1]:
#             k += 1
#         else:
#             break
#     return k


# def neighbors(word):
#     """
#     All words obtainable from 'word' by one application
#     of one of the relations (A)-(D).
#     """
#     w = list(word)
#     n = len(w)
#     out = []

#     # (A): remove [i,i]
#     for i in range(n-1):
#         if w[i] == w[i+1]:
#             out.append(tuple(w[:i] + w[i+2:]))

#     # (B): commute distant generators
#     for i in range(n-1):
#         if abs(w[i]-w[i+1]) > 1:
#             new = w[:]
#             new[i], new[i+1] = new[i+1], new[i]
#             out.append(tuple(new))

#     # (C): braid [i,i+1,i] <-> [i+1,i,i+1], i>0
#     for i in range(n-2):
#         a,b,c = w[i:i+3]

#         if a > 0:
#             if b == a+1 and c == a:
#                 new = w[:]
#                 new[i:i+3] = [a+1,a,a+1]
#                 out.append(tuple(new))

#             elif a == c-1 and b == c:
#                 # inverse move
#                 new = w[:]
#                 new[i:i+3] = [c,b-1,c]
#                 out.append(tuple(new))

#     # (D): [0,1,0,1] <-> [1,0,1,0]
#     for i in range(n-3):
#         block = w[i:i+4]

#         if block == [0,1,0,1]:
#             new = w[:]
#             new[i:i+4] = [1,0,1,0]
#             out.append(tuple(new))

#         elif block == [1,0,1,0]:
#             new = w[:]
#             new[i:i+4] = [0,1,0,1]
#             out.append(tuple(new))

#     return out


# def canonical_typeC_word(word):
#     """
#     Return the representative requested in the question.

#     Among all equivalent words:
#       1. keep only those of minimal length;
#       2. choose one with the longest strictly increasing suffix.

#     If there is still a tie, choose the lexicographically smallest
#     one to make the output deterministic.
#     """

#     start = tuple(word)

#     visited = {start}
#     Q = deque([start])

#     while Q:
#         v = Q.popleft()
#         for u in neighbors(v):
#             if u not in visited:
#                 visited.add(u)
#                 Q.append(u)

#     # shortest length
#     minlen = min(len(w) for w in visited)
#     shortest = [w for w in visited if len(w) == minlen]

#     # maximize increasing suffix length
#     best_suffix = max(increasing_suffix_length(w) for w in shortest)
#     candidates = [w for w in shortest
#                   if increasing_suffix_length(w) == best_suffix]

#     # deterministic tie-break
#     return list(min(candidates))

# # test
# t_pre = [2, 1, 1, 0, 1, 2, 1, 0, 1, 2]
# print(f"reduced word for [3,-1,-2,4] w/ max'l strictly inc'g suffix: {canonical_typeC_word(t_pre)}")

# # test 2
# t_pre = [0,1,0,1,2,1,0,1,2]
# print(f"reduced word for [-1,-2,-3] w/ max'l strictly inc'g suffix: {canonical_typeC_word(t_pre)}")

# # test 3
# t_pre = [2, 1, 2, 1, 0, 1, 2, 1, 0, 1, 2]
# print(f"reduced word for [3,-2,-1] w/ max'l strictly inc'g suffix: {canonical_typeC_word(t_pre)}")
# print("ERROR!!!! THE ABOVE EXAMPLE IS INCORRECT!!!!")

In [4]:
# finding increasing suffixes
def increasing_suffix(v):
    """
    Return the longest strictly increasing suffix of the vector v.
    """
    n = len(v)

    # Walk backwards until the increasing property fails.
    i = n - 1
    while i > 0 and v[i-1] < v[i]:
        i -= 1

    return v[i:]

# test
print(f"strictly increasing suffix of [0, 1, 2, 0, 1, 2]: {increasing_suffix([0, 1, 2, 0, 1, 2])}")

strictly increasing suffix of [0, 1, 2, 0, 1, 2]: [0, 1, 2]


In [5]:
# finding i-increasing suffixes
def i_increasing_suffix(v, i):
    """
    Return the longest strictly increasing suffix beginning at an
    occurrence of i. If multiple such occurrences exist, use the
    rightmost one. If none exist, return the empty vector.
    """
    v = list(v)
    n = len(v)

    # Check occurrences of i from right to left.
    for start in range(n - 1, -1, -1):
        if v[start] != i:
            continue

        good = True
        for j in range(start, n - 1):
            if v[j] >= v[j + 1]:
                good = False
                break

        if good:
            return v[start:]

    return []

# test
v = [0, 1, 2, 0, 1, 2]
print(f"v = {v}")
for i in range(min(v)-1,max(v)+2):
    print(f"{i}-strictly increasing suffix of v: {i_increasing_suffix(v, i)}")

v = [0, 1, 2, 0, 1, 2]
-1-strictly increasing suffix of v: []
0-strictly increasing suffix of v: [0, 1, 2]
1-strictly increasing suffix of v: [1, 2]
2-strictly increasing suffix of v: [2]
3-strictly increasing suffix of v: []


In [6]:
# given a vector t, find the maximal compatible bottom sequence b
def maxl_b(t,vec=False):
    l = len(t)-1
    # last entry of b is equal to last entry of t
    maxlB = list(range(0,l)) + [int(t[l])]
    predet = False
    # for the rest of the entries...
    for i in range(l-1,-1,-1):
        if predet == False:
            # figure out if t_i or b_{i+1} is bounding b_i from above
            maxPossible = min(int(maxlB[i+1]),int(t[i]))
            # if that upper bound is bigger than 1, check if we have a forced increase
            if maxPossible > 0 and t[i]<t[i+1]:
                maxlB[i] = min(int(maxlB[i+1])-1,int(t[i]))
            # if that upper bound is less than 1, check if we have the peak condition
            elif i>0 and maxPossible <= 0 and t[i-1]<t[i]>=t[i+1]:
                maxlB[i] = min(int(maxlB[i+1]),int(t[i]))
                # if we do have a peak and b_i and b_{i+1} are the same, then we need to make b_{i-1} at most (b_i)-1
                if maxlB[i] == maxlB[i+1]:
                    predet = True
            else:
                maxlB[i] = maxPossible
        # deal with a peak if we have
        else:
            maxlB[i] = min(int(maxlB[i+1]-1),int(t[i]))
            predet = False
    if vec == True:
        print(matrix([list(t),maxlB]))
    return maxlB

# test
print("finding a maximal compatible b for the given vector:")
v = [0, 1, 2, 0, 1, 2]
maxl_b(v,True)

finding a maximal compatible b for the given vector:
[ 0  1  2  0  1  2]
[-1 -1  0  0  1  2]


[-1, -1, 0, 0, 1, 2]

In [7]:
# printing nice tables
from sage.misc.latex import latex

def tex_table(dict1, dict2, dict3,
              key_header="i",
              dict1_header="\\Lc(w)_i",
              dict2_header=r"I_i(\mathbf{t})",
              dict3_header=r"I_i(\mathbf{b})"):

    if not (set(dict1.keys()) == set(dict2.keys()) == set(dict3.keys())):
        raise ValueError("The dictionaries must have the same keys.")

    keys = sorted(dict1.keys())

    lines = []
    lines.append(r"\[\begin{array}{|c||c|cc|}")
    lines.append(r"\hline")
    lines.append(f"{key_header} & {dict1_header} & {dict2_header} & {dict3_header} \\\\")
    lines.append(r"\hline")

    for k in keys:
        lines.append(
            f"{latex(k)} & {latex(dict1[k])} & {latex(dict2[k])} & {latex(dict3[k])} \\\\"
        )
        if k == 0:
            lines.append(r"\hline")

    lines.append(r"\hline")
    lines.append(r"\end{array}\]")

    print("\n".join(lines))

def print_dict_table(dict1, dict2, dict3,
                     key_header="i",
                     dict1_header="Lc(w)_i",
                     dict2_header="I_i(t)",
                     dict3_header="I_i(b)"):

    if not (set(dict1.keys()) == set(dict2.keys()) == set(dict3.keys())):
        raise ValueError("The dictionaries must have the same set of keys.")

    keys = sorted(dict1.keys())

    # Convert everything to strings (using Sage's latex() isn't needed here)
    rows = [[str(k), str(dict1[k]), str(dict2[k]), str(dict3[k])]
            for k in keys]

    headers = [key_header, dict1_header, dict2_header, dict3_header]

    # Compute column widths
    widths = [
        max(len(headers[j]), *(len(row[j]) for row in rows))
        for j in range(4)
    ]

    # Helper to format a row
    def fmt(row):
        return " | ".join(f"{x:<{w}}" for x, w in zip(row, widths))

    # Print table
    print(fmt(headers))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(fmt(row))

In [8]:
# some formatting helper functions
def bar_string(w):
    return "".join(
        str(x) if x >= 0 else rf"\bar{{{abs(x)}}}"
        for x in w
    )

# for printing signed permutation nicely
def split_vector_string(l):
    a, b = l
    left = ",".join(map(str, a))
    right = ",".join(map(str, b))
    return f"({left}|{right})"

# For printing Lehmer codes nicely
def coeff(v, i):
    if i > 0:
        raise ValueError("i must be nonpositive")
    elif -i >= len(v):
        return 0
    else:
        return v[i-1]

In [9]:
# Lehmer code and signed Lehmer code
def lehmer_code(w):
    """
    Return the Lehmer code of the word w.

    L_i = #{j > i : w[i] > w[j]}
    """
    n = len(w)
    return vector([
        sum(1 for j in range(i+1, n) if w[i] > w[j])
        for i in range(n)
    ])

def N(w):
    """
    Return N(w) = (N_1 > N_2 > ...), where
    N_i are the positive integers whose inverse image under the
    signed permutation w is negative.
    """
    return vector(sorted([-x for x in w if x < 0], reverse=False)) # I changed the reverse to False to make signed_Lehmer match the paper

def signed_Lehmer(w):
    return (N(w),lehmer_code(w))

# test
"""
For $u = \\overline{1}35\\overline{4}2$, then $L(u) = (1,2,2,0,0)$ and $N(u) = (4,1)$, so the signed Lehmer code $Lc(u) = (1,4\\bigg|1,1,2,0,0)$.
"""
w = [-1,3,5,-4,2]
print(f"w = {w}")
print(f"Lc(w) = {lehmer_code(w)}")
print(f"N(w) = {N(w)}")
print(f"signed Lehmer code of w: {signed_Lehmer(w)}")

w = [-1, 3, 5, -4, 2]
Lc(w) = (1, 2, 2, 0, 0)
N(w) = (1, 4)
signed Lehmer code of w: ((1, 4), (1, 2, 2, 0, 0))


In [16]:
# now the actual working function:
# Prep: Choose Type C permutation w (input as a list)
# Step 1: Use abs_and_sign_vector(w) to get flat(w) and sign(w)
# Step 2: Use t_pre = extend_list(sign,flat) to get a (not necessarily reduced) word for w
# Step 3: Use t = canonical_typeC_word(t_pre) to get a reduced word with the longest possible strictly increasing suffix
# Step 4: Use b = maxl_b(t) to get the corresponding maximal compatible bottom row
# Step 5: Use signed_Lehmer(w) to get Lc(w)
# Step 6: Use i_increasing_suffix(t,i) to get I_i(t) and I_i(b) for i in range(min(b)-1,max(b)+2)
# Step 7: Use print_dict_table to print a nice text table legible in this Jupyter Notebook
# Step 8: Use tex_table to print code for a nice TeXed table with columns i, Lc(w)_i, I_i(t), and I_i(b)

from sage.misc.latex import latex

def print_table(w,printTeX = True,printText = False):
    # Step 1
    (flat,sign) = abs_and_sign_vector(w)
    # print(f"flat: {flat}")
    # print(f"sign: {sign}")
    # STEP 1.5 NEEDS TO BE ADDED: NEED TO GO FROM FLAT TO A WORD
    flatWord = Permutation(flat).reduced_word()
    # Step 2
    t_pre = extend_list(sign,flatWord)
    # print(f"word: {t_pre}")
    # Step 3
    t = canonical_typeC_word(t_pre)
    # Step 4
    b = maxl_b(t)
    # Step 5
    Lc = signed_Lehmer(w)
    if printTeX == True:
        print(f"$w = {bar_string(w)}$; $\\mathbf{{t}} = {t}$; $\\mathbf{{b}} = {b}$")
        print("\n")
        print(f"$\\Lc(w) = {split_vector_string(Lc)}$")
        print("\n")
    if printText == True:
        print(f"w = {w}; t = {t}; b = {b}")
        print("\n")
        print(f"Lc(w) = {split_vector_string(Lc)}")
        print("\n")
    LcDict = {}
    tDict = {}
    bDict = {}
    # Step 6
    for i in range(min(b)-1,max(b)+1):
        tDict[i] = len(i_increasing_suffix(t,i))
        bDict[i] = len(i_increasing_suffix(b,i))
        if i > 0:
            LcDict[i] = Lc[1][i-1]
        else:
            LcDict[i] = coeff(Lc[0],i)
    tDict[max(b)+1] = 0
    bDict[max(b)+1] = 0
    LcDict[max(b)+1] = 0
    # Step 7
    if printText == True:
        print_dict_table(LcDict, tDict, bDict)
        print("\n")
    # Step 8
    if printTeX == True:
        tex_table(LcDict, tDict, bDict)

# test
w = [-1,-2,-3]
print_table(w,False,True)

print("\n")
print("\n")
print("\n")

# test 2
new_w = [3,-2,-1]
print_table(new_w,False,True)

w = [-1, -2, -3]; t = [0, 1, 0, 1, 2, 1, 0, 1, 2]; b = [-2, -1, -1, -1, 0, 0, 0, 1, 2]


Lc(w) = (1,2,3|2,1,0)


i  | Lc(w)_i | I_i(t) | I_i(b)
---+---------+--------+-------
-3 | 0       | 0      | 0     
-2 | 1       | 0      | 0     
-1 | 2       | 0      | 0     
0  | 3       | 3      | 3     
1  | 2       | 2      | 2     
2  | 1       | 1      | 1     
3  | 0       | 0      | 0     








w = [3, -2, -1]; t = [0, 1, 2, 0, 1]; b = [-1, -1, 0, 0, 1]


Lc(w) = (1,2|2,0,0)


i  | Lc(w)_i | I_i(t) | I_i(b)
---+---------+--------+-------
-2 | 0       | 0      | 0     
-1 | 1       | 0      | 0     
0  | 2       | 2      | 2     
1  | 2       | 1      | 1     
2  | 0       | 0      | 0     




In [17]:
# generating all Type C permutations for a given n
from sage.combinat.permutation import Permutations

# from GPT-5.5 on 7/12/2026
from itertools import product

def all_signs(v):
    """
    Given a vector v, return all vectors obtained by independently
    choosing either v_i or -v_i for each entry.
    """
    return [
        [s * x for s, x in zip(signs, v)]
        for signs in product([1, -1], repeat=len(v))
    ]

# function to generate type c permutations
def typec_perms(n,toPrint=False):
    base = Permutations(n)
    final = []
    for perm in base:
        final = final + all_signs(perm)
    if toPrint == True:
        i = 1
        for perm in final:
            print(f"{i}: {perm}")
            i += 1
        print(f"{n}!*(2^{n}) = {factorial(n)*(2^n)}")
    return final

# test
typec_perms(3,False)

[[1, 2, 3],
 [1, 2, -3],
 [1, -2, 3],
 [1, -2, -3],
 [-1, 2, 3],
 [-1, 2, -3],
 [-1, -2, 3],
 [-1, -2, -3],
 [1, 3, 2],
 [1, 3, -2],
 [1, -3, 2],
 [1, -3, -2],
 [-1, 3, 2],
 [-1, 3, -2],
 [-1, -3, 2],
 [-1, -3, -2],
 [2, 1, 3],
 [2, 1, -3],
 [2, -1, 3],
 [2, -1, -3],
 [-2, 1, 3],
 [-2, 1, -3],
 [-2, -1, 3],
 [-2, -1, -3],
 [2, 3, 1],
 [2, 3, -1],
 [2, -3, 1],
 [2, -3, -1],
 [-2, 3, 1],
 [-2, 3, -1],
 [-2, -3, 1],
 [-2, -3, -1],
 [3, 1, 2],
 [3, 1, -2],
 [3, -1, 2],
 [3, -1, -2],
 [-3, 1, 2],
 [-3, 1, -2],
 [-3, -1, 2],
 [-3, -1, -2],
 [3, 2, 1],
 [3, 2, -1],
 [3, -2, 1],
 [3, -2, -1],
 [-3, 2, 1],
 [-3, 2, -1],
 [-3, -2, 1],
 [-3, -2, -1]]

# actual data collection

In [18]:
%%time
perms = typec_perms(1,False)
for perm in perms[1:]:
    print_table(perm)
    print("\n")
    print("\n")
    print("\n")

$w = \bar{1}$; $\mathbf{t} = [0]$; $\mathbf{b} = [0]$


$\Lc(w) = (1|0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 1 & 1 & 1 \\
\hline
1 & 0 & 0 & 0 \\
\hline
\end{array}\]






CPU times: user 0 ns, sys: 5.46 ms, total: 5.46 ms
Wall time: 5.37 ms


In [19]:
%%time
perms = typec_perms(2,False)
for perm in perms[1:]:
    print_table(perm)
    print("\n")
    print("\n")
    print("\n")

$w = 1\bar{2}$; $\mathbf{t} = [1, 0, 1]$; $\mathbf{b} = [0, 0, 1]$


$\Lc(w) = (2|1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 2 & 2 & 2 \\
\hline
1 & 1 & 1 & 1 \\
2 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = \bar{1}2$; $\mathbf{t} = [0]$; $\mathbf{b} = [0]$


$\Lc(w) = (1|0,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 1 & 1 & 1 \\
\hline
1 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = \bar{1}\bar{2}$; $\mathbf{t} = [0, 1, 0, 1]$; $\mathbf{b} = [-1, 0, 0, 1]$


$\Lc(w) = (1,2|1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-2 & 0 & 0 & 0 \\
-1 & 1 & 0 & 0 \\
0 & 2 & 2 & 2 \\
\hline
1 & 1 & 1 & 1 \\
2 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = 21$; $\mathbf{t} = [1]$; $\mathbf{b} = [1]$


$\Lc(w) = (|1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & 

In [20]:
%%time
perms = typec_perms(3,False)
for perm in perms[1:]:
    print_table(perm)
    print("\n")
    print("\n")
    print("\n")

$w = 12\bar{3}$; $\mathbf{t} = [2, 1, 0, 1, 2]$; $\mathbf{b} = [0, 0, 0, 1, 2]$


$\Lc(w) = (3|1,1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 3 & 3 & 3 \\
\hline
1 & 1 & 2 & 2 \\
2 & 1 & 1 & 1 \\
3 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = 1\bar{2}3$; $\mathbf{t} = [1, 0, 1]$; $\mathbf{b} = [0, 0, 1]$


$\Lc(w) = (2|1,0,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 2 & 2 & 2 \\
\hline
1 & 1 & 1 & 1 \\
2 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = 1\bar{2}\bar{3}$; $\mathbf{t} = [1, 0, 1, 2, 1, 0, 1, 2]$; $\mathbf{b} = [-1, -1, -1, 0, 0, 0, 1, 2]$


$\Lc(w) = (2,3|2,1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-2 & 0 & 0 & 0 \\
-1 & 2 & 0 & 0 \\
0 & 3 & 3 & 3 \\
\hline
1 & 2 & 2 & 2 \\
2 & 1 & 1 & 1 \\
3 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = \bar{1}23$; $\mathbf{t}

In [21]:
%%time
perms = typec_perms(4,False)
for perm in perms[1:]:
    print_table(perm)
    print("\n")
    print("\n")
    print("\n")

$w = 123\bar{4}$; $\mathbf{t} = [3, 2, 1, 0, 1, 2, 3]$; $\mathbf{b} = [0, 0, 0, 0, 1, 2, 3]$


$\Lc(w) = (4|1,1,1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 4 & 4 & 4 \\
\hline
1 & 1 & 3 & 3 \\
2 & 1 & 2 & 2 \\
3 & 1 & 1 & 1 \\
4 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = 12\bar{3}4$; $\mathbf{t} = [2, 1, 0, 1, 2]$; $\mathbf{b} = [0, 0, 0, 1, 2]$


$\Lc(w) = (3|1,1,0,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-1 & 0 & 0 & 0 \\
0 & 3 & 3 & 3 \\
\hline
1 & 1 & 2 & 2 \\
2 & 1 & 1 & 1 \\
3 & 0 & 0 & 0 \\
\hline
\end{array}\]






$w = 12\bar{3}\bar{4}$; $\mathbf{t} = [2, 1, 0, 1, 2, 3, 2, 1, 0, 1, 2, 3]$; $\mathbf{b} = [-1, -1, -1, -1, -1, 0, 0, 0, 0, 1, 2, 3]$


$\Lc(w) = (3,4|2,2,1,0)$


\[\begin{array}{|c||c|cc|}
\hline
i & \Lc(w)_i & I_i(\mathbf{t}) & I_i(\mathbf{b}) \\
\hline
-2 & 0 & 0 & 0 \\
-1 & 3 & 0 & 0 \\
0 & 4 & 4 & 4 \\
\hline
1 & 2 & 3 & 3